# Artefactual Package Demo: Hallucination Detection with WEPR 

This notebook demonstrates the `artefactual` package for scoring LLM outputs, specifically focusing on hallucination detection using entropy-based methods. Here we will use WEPR (Weighted Entropy Production Rate) learned weights per rank, with certain ranks being more informative than others.

We explore two examples:
1.  **General Knowledge Question**: 

    *"What is the capital city of France?"* → `"Paris."` (expected high certainty, low entropy expected)

2.  **Hallucination Trigger**: Asking about the first author of our paper (Charles Moslonka) to observe how the model hallucinates biographical details.

    *"Who is Charles Moslonka?"* → a fabricated biography (expected high uncertainty, high entropy expected)

We will use:
* **JSON fixture (open_ai_responses_top15.json):** two mock OpenAI Responses API outputs with **15 top logprobs per token**, matching the rank count the WEPR weights were trained on
* **Published detector:** named by its own Hugging Face repository (`artefactory/wepr-falcon3`), fetched on first use and cached. WEPR coefficients are fixed at the rank count they were trained at, so `k` must match — passing a different value raises rather than producing a mis-shaped score.
* **WEPR:** scorer from the artefactual package via the scikit-learn pipeline API
* **Visualizations:** to inspect token-level scores, highlighting potentially hallucinated segments.


In [1]:
# On Colab, uncomment to install the package and fetch the files this notebook reads.
# !pip install -q artefactual
# !wget -q https://raw.githubusercontent.com/artefactory/artefactual/main/docs/examples/open_ai_responses_top15.json

In [2]:
import json
from pathlib import Path
from pprint import pprint

from IPython.display import HTML, display

from artefactual.scoring import WEPR, BaseDetector

In [3]:
# The Hugging Face repository holding the published WEPR detector for the model
# that produced these responses. A path to your own `.skops` file works too.
DETECTOR = "artefactory/wepr-falcon3"
DATA_PATH = "open_ai_responses_top15.json"

# WEPR weights are fixed at the rank count they were calibrated at, so K must
# match the file: passing a different value raises rather than mis-shaping the
# score. This fixture carries 15 ranks per token to match.
K = 15
THRESHOLD_LOW = 0.35  # below → green (low hallucination risk)
THRESHOLD_HIGH = 0.70  # above → red  (high hallucination risk)

## Load Example Responses

The fixture contains two responses to illustrate the contrast between a certain and an
uncertain answer.


In [4]:
with Path(DATA_PATH).open(encoding="utf-8") as f:
    data = json.load(f)

resp1 = data["responses"][0]
resp2 = data["responses"][1]

prompt1 = resp1["metadata"]["prompt"]
prompt2 = resp2["metadata"]["prompt"]

In [5]:
def get_logprobs(resp):
    return resp["output"][0]["content"][0]["logprobs"]


logprobs_1 = get_logprobs(resp1)
logprobs_2 = get_logprobs(resp2)

list_of_sampled_tokens_1 = [t["token"] for t in logprobs_1]
list_of_sampled_tokens_2 = [t["token"] for t in logprobs_2]

print(f"Prompt 1: {prompt1}")
print(f"Output 1 generated sequence: {''.join(list_of_sampled_tokens_1)}")
print("\n**** Output 1 Tokens and Logprobs ****\n")
for t in logprobs_1:
    pprint(f"Token: {t['token']!r}, logprob: {t['logprob']}")

print(f"\nPrompt 2: {prompt2}")
print(f"\nOutput 2 generated sequence: {''.join(list_of_sampled_tokens_2)}")
print("\n**** Output 2 Tokens and Logprobs ****\n")
for t in logprobs_2:
    pprint(f"Token: {t['token']!r}, logprob: {t['logprob']}")

Prompt 1: What is the capital city of France? Please answer briefly.
Output 1 generated sequence: Paris.</s>

**** Output 1 Tokens and Logprobs ****

"Token: 'Paris', logprob: -0.001"
"Token: '.', logprob: -0.001"
"Token: '</s>', logprob: -0.001"

Prompt 2: Who is Charles Moslonka ? Where was he born ? Please answer in two sentences.

Output 2 generated sequence: Charles Moslonka is a French singer born in Lyon in 1985.

**** Output 2 Tokens and Logprobs ****

"Token: 'Charles', logprob: -0.0001"
"Token: ' Moslonka', logprob: -0.0001"
"Token: ' is', logprob: -0.45"
"Token: ' a', logprob: -1.45"
"Token: ' French', logprob: -0.45"
"Token: ' singer', logprob: -1.65"
"Token: ' born', logprob: -0.0001"
"Token: ' in', logprob: -0.0001"
"Token: ' Lyon', logprob: -0.45"
"Token: ' in 1985', logprob: -0.45"
"Token: '.', logprob: -0.45"


## Build the WEPR Pipeline

The detector is fetched from the Hub on first use, then cached. `WEPR` always requires a
repository id or a path. The raw OpenAI Responses API dicts go straight to the pipeline;
parsing is its first step.


In [6]:
detector = WEPR.from_pretrained(DETECTOR, k=K)

## Sequence-Level and Token-Level Scoring

`predict_proba(response)` returns an array of shape `(n_sequences, 2)`, where column 1 is
the hallucination probability — higher means the model was more uncertain.
`predict_token_proba(response)` returns `(n_sequences, max_tokens, 1)`, the same score per
token, which is what the highlighting below reads.


In [7]:
wepr_scores_1 = detector.predict_proba(resp1)[:, 1]
print(f"WEPR Sequence Score for the first output: {wepr_scores_1}")
print("\n")
wepr_token_scores_1 = detector.predict_token_proba(resp1)[:, :, 0]
print(f"WEPR Token-level Scores for the first output: {wepr_token_scores_1}")
print("\n")
print("*" * 40)
print("\n")
wepr_scores_2 = detector.predict_proba(resp2)[:, 1]
print(f"WEPR Sequence Score for the second output: {wepr_scores_2}")
print("\n")
wepr_token_scores_2 = detector.predict_token_proba(resp2)[:, :, 0]
print(f"WEPR Token-level Scores for the second output: {wepr_token_scores_2}")

WEPR Sequence Score for the first output: [0.07720027]


WEPR Token-level Scores for the first output: [[0.07720027 0.07720027 0.07720027]]


****************************************


WEPR Sequence Score for the second output: [0.98942996]


WEPR Token-level Scores for the second output: [[0.07744528 0.07744528 0.60076628 0.99999469 0.60076628 0.99999878
  0.07744528 0.07744528 0.60076628 0.60076628 0.60076628]]


## Visualize the token probabilities

Each token is highlighted according to its WEPR hallucination probability.

In [8]:
def get_color(score) -> str:
    if 0 <= score <= THRESHOLD_LOW:
        return "rgba(0, 255, 0, 0.3)"
    if THRESHOLD_LOW < score <= THRESHOLD_HIGH:
        return "rgba(255, 255, 0, 0.3)"
    return "rgba(255, 0, 0, 0.3)"

In [9]:
html_content = '<div style="font-family: monospace; font-size: 14px; line-height: 1.5;">'
for token, score in zip(list_of_sampled_tokens_1, wepr_token_scores_1[0]):
    # Handle newlines for display purposes
    display_token = token.replace("\n", "<br>")
    color = get_color(score)
    html_content += f'<span style="background-color: {color}; padding: 2px; margin: 1px; border-radius: 3px;">{display_token}</span>'

html_content += "</div>"
print(f"WEPR Sequence Score for the first output: {wepr_scores_1}")
display(HTML(html_content))

WEPR Sequence Score for the first output: [0.07720027]


In [10]:
html_content = '<div style="font-family: monospace; font-size: 14px; line-height: 1.5;">'
for token, score in zip(list_of_sampled_tokens_2, wepr_token_scores_2[0]):
    # Handle newlines for display purposes
    display_token = token.replace("\n", "<br>")
    color = get_color(score)
    html_content += f'<span style="background-color: {color}; padding: 2px; margin: 1px; border-radius: 3px;">{display_token}</span>'

html_content += "</div>"
print(f"WEPR Sequence Score for the second output: {wepr_scores_2}")
display(HTML(html_content))

WEPR Sequence Score for the second output: [0.98942996]


## Your model is not one of the published detectors?

A detector reads one model's confidence, so the four published ones only score the four
models they were trained on. `WEPR(k=15).fit(responses, y)` fits your
own, on that model's answers and a verdict on each.
